# Farm Boundary Delineation with the Segment Anything Model

**Study area:** Loni, Maharashtra, India — 0.5 m drone orthomosaic
**CRS:** WGS 84 / UTM zone 43N (EPSG:32643)

This notebook applies the Segment Anything Model (SAM) in automatic mask
generation mode to a 0.5 m orthomosaic, then filters, vectorises and
characterises the resulting parcel boundaries. It accompanies Section 3.3.3
of the thesis and produces Figure 3.6.

**Pipeline**

1. Fetch imagery and prepare a 3-band RGB Cloud-Optimised GeoTIFF
2. Automatic mask generation with SAM (ViT-H), tiled at native resolution
3. Vectorise masks to polygons
4. Area-threshold sensitivity analysis and filtering
5. Figure 3.6 and export of results

**Runtime:** set *Runtime → Change runtime type → T4 GPU* before running.
Step 3 takes several minutes on a GPU and is impractical on CPU.

---

*Citation:* if you use this notebook or the derived boundaries, please cite
the thesis and the archived dataset DOI listed in the repository README.

## 1. Setup

In [ ]:
%%capture
# segment-geospatial pulls in torch + SAM; this takes a few minutes.
if 'google.colab' in str(get_ipython()):
    !pip install segment-geospatial rioxarray geopandas seaborn

In [ ]:
import os
import numpy as np
import geopandas as gpd
import rioxarray as rxr
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from matplotlib.colors import ListedColormap
from samgeo import SamGeo

data_folder, output_folder = 'data', 'output'
os.makedirs(data_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

sns.set_theme(style='white', context='paper')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 11,
    'axes.titleweight': 'semibold',
    'axes.titlelocation': 'left',
    'savefig.facecolor': 'white',
})

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Set Runtime > Change runtime type > T4 GPU.')

## 2. Data

The orthomosaic is too large for a plain GitHub file, so it is distributed as
a **GitHub release asset** (up to 2 GB per file) and archived on Zenodo with a
DOI. Edit `DATA_URL` to point at your release.

In [ ]:
import requests

DATA_URL = (
    'https://github.com/<user>/<repo>/releases/download/v1.0/loni_sub_0p5m.tif'
)
SOURCE_NAME = 'loni_sub_0p5m.tif'


def download(url, folder=data_folder):
    """Idempotent download: skip if the file already exists."""
    filename = os.path.join(folder, os.path.basename(url))
    if not os.path.exists(filename):
        with requests.get(url, stream=True, allow_redirects=True) as r:
            r.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print('Downloaded', filename)
    else:
        print('Already present:', filename)
    return filename


source_path = download(DATA_URL)
# If you are uploading manually instead, comment out the line above and use:
# source_path = os.path.join(data_folder, SOURCE_NAME)

### 2.1 Prepare a 3-band RGB COG

The orthomosaic carries a fourth alpha band. SAM expects exactly three
channels, so the alpha is dropped and the result rewritten as a
Cloud-Optimised GeoTIFF. `gdalinfo -checksum` reads every tile, which also
verifies the file is not truncated.

In [ ]:
image_path = os.path.join(data_folder, 'loni_rgb.tif')

if not os.path.exists(image_path):
    !gdal_translate "{source_path}" "{image_path}" -b 1 -b 2 -b 3 -of COG -co COMPRESS=DEFLATE -co NUM_THREADS=ALL_CPUS

!gdalinfo -checksum "{image_path}" | head -n 5

In [ ]:
image = rxr.open_rasterio(image_path)

bands, height, width = image.shape
res_x, res_y = abs(image.rio.resolution()[0]), abs(image.rio.resolution()[1])
extent_m = (width * res_x, height * res_y)
area_km2 = extent_m[0] * extent_m[1] / 1e6

print(f'Bands      : {bands}  ({image.dtype})')
print(f'Size       : {width} x {height} px')
print(f'Resolution : {res_x:.3f} x {res_y:.3f} m')
print(f'Extent     : {extent_m[0]/1000:.3f} x {extent_m[1]/1000:.3f} km'
      f'  = {area_km2:.3f} km2')
print(f'CRS        : {image.rio.crs}')

## 3. Segmentation with SAM

In [ ]:
sam = SamGeo(
    model_type='vit_h',
    sam_kwargs=None,
)

In [ ]:
mask_path = os.path.join(output_folder, 'segment.tif')

# batch=True tiles the image and runs SAM at native resolution, instead of
# downsampling the whole scene to the model's 1024 px input. unique=True
# writes a distinct integer id per mask so the raster can be polygonised.
if not os.path.exists(mask_path):
    sam.generate(
        image_path,
        mask_path,
        batch=True,
        foreground=True,
        erosion_kernel=(3, 3),
        mask_multiplier=255,
        unique=True,
    )
print('mask:', mask_path)

## 4. Vectorise

In [ ]:
shapefile_path = os.path.join(output_folder, 'segment.shp')

# simplify_tolerance in CRS units; 0.5 matches the pixel grid and removes
# staircase artefacts without displacing boundaries beyond one pixel.
sam.tiff_to_shp(mask_path, shapefile_path, simplify_tolerance=0.5)

gdf = gpd.read_file(shapefile_path)
gdf['geometry'] = gdf.geometry.buffer(0)          # repair invalid rings
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty]
gdf['area_m2'] = gdf.geometry.area
print(f'{len(gdf)} raw polygons')

## 5. Area-threshold sensitivity analysis

SAM in automatic mode is strongly over-segmenting: it fires on crop rows,
shadow lines and canopy texture as well as on parcel bunds. The polygon size
distribution is bimodal — a large spike at one to a few pixels, then a
separate mode at plausible plot sizes. The floor is chosen at the elbow of
the count-versus-threshold curve rather than arbitrarily.

In [ ]:
floors = [0, 10, 50, 100, 200, 500, 1000]
MAX_AREA = 20000       # m2; excludes the scene-wide background polygon

rows = []
for lo in floors:
    n = ((gdf['area_m2'] >= lo) & (gdf['area_m2'] <= MAX_AREA)).sum()
    rows.append({'floor_m2': lo, 'n_polygons': int(n)})

sensitivity = gpd.pd.DataFrame(rows)
print(sensitivity.to_string(index=False))
print()
print(gdf['area_m2'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(gdf['area_m2'], bins=120, log=True, color=sns.color_palette('colorblind')[0])
axes[0].set_xlabel('Polygon area (m$^2$)')
axes[0].set_ylabel('Count (log)')
axes[0].set_title('(a) Raw polygon size distribution')

axes[1].plot(sensitivity['floor_m2'], sensitivity['n_polygons'],
             marker='o', color=sns.color_palette('colorblind')[0])
axes[1].axvline(100, color=sns.color_palette('colorblind')[3],
                ls='--', lw=1.2, label='chosen floor: 100 m$^2$')
axes[1].set_xlabel('Minimum area threshold (m$^2$)')
axes[1].set_ylabel('Polygons retained')
axes[1].set_title('(b) Sensitivity to the area floor')
axes[1].legend(fontsize=8)

sns.despine(fig=fig)
fig.tight_layout()
fig.savefig(os.path.join(output_folder, 'figure_area_sensitivity.pdf'),
            bbox_inches='tight')
plt.show()

In [ ]:
MIN_AREA = 100         # m2; 0.01 ha, below the smallest plausible plot

cleaned_gdf = gdf[(gdf['area_m2'] >= MIN_AREA) &
                  (gdf['area_m2'] <= MAX_AREA)].copy()

coverage = cleaned_gdf['area_m2'].sum() / (area_km2 * 1e6)
print(f'{len(gdf)} -> {len(cleaned_gdf)} polygons')
print(f'median area   : {cleaned_gdf["area_m2"].median():,.0f} m2')
print(f'mean area     : {cleaned_gdf["area_m2"].mean():,.0f} m2')
print(f'scene coverage: {coverage:.1%}')
print(f'density       : {len(cleaned_gdf)/area_km2:.0f} polygons per km2')

## 6. Figure 3.6

Three panels: the input orthomosaic, the raw SAM masks, and the filtered and
vectorised polygons.

In [ ]:
def rgb_extent(da):
    """(H, W, 3) array plus a matplotlib extent tuple in CRS units."""
    arr = da.values[:3]
    arr = np.transpose(arr, (1, 2, 0))
    if arr.dtype != np.uint8:
        lo, hi = np.nanpercentile(arr, [2, 98])
        arr = np.clip((arr - lo) / max(hi - lo, 1e-9), 0, 1)
    minx, miny, maxx, maxy = da.rio.bounds()
    return arr, (minx, maxx, miny, maxy)


def random_cmap(n=256, seed=42):
    rng = np.random.default_rng(seed)
    colors = rng.random((n, 3)) * 0.75 + 0.25
    colors[0] = [0, 0, 0]
    return ListedColormap(colors)


def frame(ax, title, extent):
    ax.set_title(title)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.6); s.set_color('0.6')

In [ ]:
rgb, extent = rgb_extent(image)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
ax_a, ax_b, ax_c = axes

ax_a.imshow(rgb, extent=extent, interpolation='nearest')
frame(ax_a, '(a) 0.5 m orthomosaic', extent)

mask = rxr.open_rasterio(mask_path).squeeze()
m = mask.values.astype(float)
m = np.where(m > 0, (m % 255) + 1, 0)
ax_b.imshow(m, extent=extent, cmap=random_cmap(), interpolation='nearest')
frame(ax_b, f'(b) Raw SAM masks (n = {len(gdf):,})', extent)

ax_c.imshow(rgb, extent=extent, interpolation='nearest', alpha=0.75)
cleaned_gdf.plot(ax=ax_c, facecolor='none',
                 edgecolor=sns.color_palette('colorblind')[0], linewidth=0.7)
frame(ax_c, f'(c) Filtered polygons (n = {len(cleaned_gdf):,})', extent)

sns.despine(fig=fig, left=True, bottom=True)
fig.tight_layout()
fig.savefig(os.path.join(output_folder, 'figure_3_6.pdf'),
            bbox_inches='tight', pad_inches=0.15)
fig.savefig(os.path.join(output_folder, 'figure_3_6.png'), dpi=300,
            bbox_inches='tight', pad_inches=0.15)
plt.show()

## 7. Export

In [ ]:
import zipfile

out_shp = os.path.join(output_folder, 'segmentation_results.shp')
cleaned_gdf.to_file(out_shp)

base = os.path.splitext(out_shp)[0]
zip_path = os.path.join(output_folder, 'segmentation_results.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
        p = base + ext
        if os.path.exists(p):
            zf.write(p, os.path.basename(p))

# GeoPackage keeps full field names and one file instead of six.
cleaned_gdf.to_file(os.path.join(output_folder, 'segmentation_results.gpkg'),
                    driver='GPKG')
sensitivity.to_csv(os.path.join(output_folder, 'area_sensitivity.csv'),
                   index=False)

print('written to', output_folder)

## Next steps

Two stages of the chain described in Section 3.3.3 are not yet implemented
here and are the natural continuation of this notebook:

- **Contrast-based merging.** Build a region-adjacency graph over touching
  polygons, compute per-polygon spectral and textural statistics, and dissolve
  neighbour pairs whose cross-boundary contrast falls below a threshold. This
  targets the dominant failure mode — a single managed field split along an
  internal furrow or shadow line.
- **Cadastral reconciliation.** Intersect the derived polygons with the
  survey-number layer so each inherits a survey attribution, and record the
  agreement between layers. Survey numbers containing several distinct
  segments are the multi-cropped parcels of interest.

Evaluation against a manually digitised reference set (IoU, boundary F1,
over- and under-segmentation rates) follows once both are in place.